# EEG Braindecode — Baseline su Segnale Raw (4/5 Classi Semantiche)

Questo notebook testa **5 modelli end-to-end** di [Braindecode](https://braindecode.org) sul segnale EEG grezzo,
usando il clustering semantico a 4 e 5 classi invece delle 110 parole originali.

## Modelli testati
| Modello | Architettura | Parametri | Note |
|---------|-------------|-----------|------|
| **EEGNet** | CNN compatta | ~2.8K | Baseline leggero, universale |
| **ShallowFBCSPNet** | CNN superficiale + freq | ~98K | Ispira a FBCSP classico |
| **Deep4Net** | CNN profonda | ~261K | Baseline convoluzionale solido |
| **EEGConformer** | CNN + Transformer | ~429K | Cattura pattern locali e globali |
| **ATCNet** | Attention + TCN | ~44K | Sliding window con attenzione |

> **Nota CBraMod**: il modello richiede Python ≥ 3.11; l'ambiente attuale è 3.10.
> Aggiornare Python per includerlo in esperimenti futuri.

## Setup dati
- Input: segnale EEG grezzo `(batch, 59, 384)` — 59 canali × 384 campioni a 256 Hz (~1.5s)
- Label: cluster semantici a **4 classi** (azioni, cognitivo, emozioni, oggetti) o **5 classi**
- Valutazione: **subject-specific** (train/val/test sullo stesso soggetto)

Data: 2026-03-12

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # fix OpenMP su macOS

import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

from braindecode.models import EEGNet, EEGConformer, Deep4Net, ShallowFBCSPNet, ATCNet

# Device: MPS (Apple Silicon) > CUDA > CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device:", device)
print("torch:", torch.__version__)
import braindecode; print("braindecode:", braindecode.__version__)

In [ ]:
# ============================================================
# CONFIGURAZIONE
# ============================================================

project_root = Path("/Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech")

META_CSV    = project_root / "data" / "interim" / "eeg_metadata.csv"
L2C4_JSON  = project_root / "data" / "interim" / "labelid2cluster_4.json"
L2C5_JSON  = project_root / "data" / "interim" / "labelid2cluster_5.json"
ELOC_PATH  = project_root / "src" / "io" / "ebneuro.locs"

# Parametri EEG
N_CHANS  = 59      # canali dopo rimozione A1, A2
N_TIMES  = 384     # campioni a 256 Hz (~1.5s)
SFREQ    = 256

# Training
BATCH_SIZE  = 64
MAX_EPOCHS  = 100
PATIENCE    = 15
LR          = 1e-3
WEIGHT_DECAY = 1e-4

# Soggetti da testare in subject-specific (0-based)
TEST_SUBJECTS = [0, 1, 2, 3, 4]

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("Config OK")

In [ ]:
# ============================================================
# CARICAMENTO METADATA E CLUSTER MAPPING
# ============================================================

meta = pd.read_csv(META_CSV)
meta["subject_id"] = meta["subject_id"].astype(str).str.zfill(2)

with open(L2C4_JSON, "r") as f:
    labelid2cluster_4 = {int(k): int(v) for k, v in json.load(f).items()}
with open(L2C5_JSON, "r") as f:
    labelid2cluster_5 = {int(k): int(v) for k, v in json.load(f).items()}

# Indici dei canali da mantenere (rimuove A1 idx=0, A2 idx=7 dal file .locs)
def read_eloc_names(path):
    names = []
    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                names.append(parts[3])
    return names[:61]  # H5 ha 61 canali

ch_names_61 = read_eloc_names(ELOC_PATH)
EXCLUDE = {"A1", "A2"}
keep_idx = [i for i, n in enumerate(ch_names_61) if n not in EXCLUDE]
keep_names = [ch_names_61[i] for i in keep_idx]

assert len(keep_idx) == N_CHANS, f"Atteso {N_CHANS} canali, trovato {len(keep_idx)}"

print(f"Meta rows: {len(meta)} | soggetti: {meta['subject_id'].nunique()}")
print(f"Canali mantenuti: {len(keep_idx)} ({keep_names[:5]}...)")
print(f"Cluster 4 — distribuzione parole: { {k: list(labelid2cluster_4.values()).count(k) for k in range(4)} }")
print(f"Cluster 5 — distribuzione parole: { {k: list(labelid2cluster_5.values()).count(k) for k in range(5)} }")

In [ ]:
# ============================================================
# DATASET: caricamento lazy da H5
# ============================================================

class RawEEGDataset(Dataset):
    """
    Dataset che carica epoche EEG grezze da file H5.
    Input: (n_channels, n_times) per ogni trial.
    Normalizzazione per-canale (zero mean, unit variance) stimata su train set.
    """
    def __init__(self, records, keep_idx, labelid2cluster, mean=None, std=None):
        """
        records: lista di dict con chiavi 'path_h5', 'epoch_idx', 'label_idx'
        keep_idx: indici canali da mantenere
        labelid2cluster: mapping label_id -> cluster_id
        mean, std: (n_chans, 1) per normalizzazione; se None si calcola da questo split
        """
        self.records = records
        self.keep_idx = keep_idx
        self.labelid2cluster = labelid2cluster
        self.mean = mean
        self.std = std

        if mean is None or std is None:
            self._compute_stats()

    def _compute_stats(self):
        """Calcola mean/std su un campione casuale per normalizzazione."""
        sample_size = min(500, len(self.records))
        indices = np.random.choice(len(self.records), sample_size, replace=False)
        all_x = []
        for i in indices:
            r = self.records[i]
            with h5py.File(r["path_h5"], "r") as f:
                x = f["data"][int(r["epoch_idx"])]  # (61, T)
            x = x[self.keep_idx, :].astype(np.float32)  # (59, T)
            all_x.append(x)
        all_x = np.stack(all_x, axis=0)  # (N, 59, T)
        self.mean = all_x.mean(axis=(0, 2), keepdims=True).squeeze(0)[:, :1].astype(np.float32)  # (59, 1)
        self.std  = all_x.std(axis=(0, 2), keepdims=True).squeeze(0)[:, :1].astype(np.float32) + 1e-6

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        with h5py.File(r["path_h5"], "r") as f:
            x = f["data"][int(r["epoch_idx"])]  # (61, T)
        x = x[self.keep_idx, :].astype(np.float32)  # (59, T)
        x = (x - self.mean) / self.std  # normalizzazione per canale
        label = self.labelid2cluster[int(r["label_idx"])]
        return torch.from_numpy(x), torch.tensor(label, dtype=torch.long)


def make_subject_splits(meta_df, subject_id, labelid2cluster,
                        train_frac=0.6, val_frac=0.2, seed=42):
    """Split 60/20/20 per un singolo soggetto."""
    df_s = meta_df[meta_df["subject_id"] == str(subject_id).zfill(2)].copy()
    records = df_s[["path_h5", "epoch_idx", "label_idx"]].to_dict("records")

    np.random.seed(seed)
    idx = np.random.permutation(len(records))
    n_train = int(len(idx) * train_frac)
    n_val   = int(len(idx) * val_frac)

    r_train = [records[i] for i in idx[:n_train]]
    r_val   = [records[i] for i in idx[n_train:n_train + n_val]]
    r_test  = [records[i] for i in idx[n_train + n_val:]]

    ds_train = RawEEGDataset(r_train, keep_idx, labelid2cluster)
    ds_val   = RawEEGDataset(r_val,   keep_idx, labelid2cluster, ds_train.mean, ds_train.std)
    ds_test  = RawEEGDataset(r_test,  keep_idx, labelid2cluster, ds_train.mean, ds_train.std)

    return ds_train, ds_val, ds_test


print("Dataset classe OK")

In [ ]:
# ============================================================
# FACTORY MODELLI
# ============================================================

def build_model(name, n_outputs):
    """Istanzia un modello braindecode per i nostri dati EEG."""
    kw = dict(n_chans=N_CHANS, n_outputs=n_outputs, n_times=N_TIMES, sfreq=SFREQ)
    if name == "EEGNet":
        return EEGNet(**kw, final_conv_length="auto")
    elif name == "ShallowFBCSPNet":
        return ShallowFBCSPNet(**kw, final_conv_length="auto")
    elif name == "Deep4Net":
        return Deep4Net(**kw, final_conv_length="auto")
    elif name == "EEGConformer":
        return EEGConformer(n_chans=N_CHANS, n_outputs=n_outputs,
                            n_times=N_TIMES, sfreq=SFREQ, final_fc_length="auto")
    elif name == "ATCNet":
        return ATCNet(n_chans=N_CHANS, n_outputs=n_outputs,
                      input_window_seconds=N_TIMES / SFREQ, sfreq=SFREQ)
    else:
        raise ValueError(f"Modello sconosciuto: {name}")


MODEL_NAMES = ["EEGNet", "ShallowFBCSPNet", "Deep4Net", "EEGConformer", "ATCNet"]

# Stampa parametri per ogni modello
print(f"{'Modello':<18} {'Parametri':>12}")
print("-" * 32)
for name in MODEL_NAMES:
    m = build_model(name, n_outputs=4)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"{name:<18} {n_params:>12,}")

In [ ]:
# ============================================================
# TRAINING E VALUTAZIONE
# ============================================================

def train_model(model, ds_train, ds_val, n_epochs=MAX_EPOCHS, patience=PATIENCE,
                lr=LR, weight_decay=WEIGHT_DECAY, batch_size=BATCH_SIZE, device=device):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    criterion = nn.CrossEntropyLoss()

    loader_tr = DataLoader(ds_train, batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=False)
    loader_va = DataLoader(ds_val,   batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)

    best_val_acc = 0.0
    best_state  = None
    patience_cnt = 0
    history = defaultdict(list)

    for epoch in range(n_epochs):
        # --- train ---
        model.train()
        train_loss, train_correct, n_total = 0.0, 0, 0
        for x, y in loader_tr:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss    += loss.item() * len(y)
            train_correct += (logits.argmax(1) == y).sum().item()
            n_total       += len(y)
        scheduler.step()

        # --- val ---
        model.eval()
        val_correct, val_total = 0, 0
        ys_v, ps_v = [], []
        with torch.no_grad():
            for x, y in loader_va:
                x, y = x.to(device), y.to(device)
                preds = model(x).argmax(1)
                val_correct += (preds == y).sum().item()
                val_total   += len(y)
                ys_v.extend(y.cpu().tolist())
                ps_v.extend(preds.cpu().tolist())

        train_acc = train_correct / n_total
        val_acc   = val_correct / val_total
        val_bacc  = balanced_accuracy_score(ys_v, ps_v)

        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["val_bacc"].append(val_bacc)
        history["train_loss"].append(train_loss / n_total)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                break

    model.load_state_dict(best_state)
    return model, dict(history), epoch + 1


def evaluate(model, ds, batch_size=BATCH_SIZE, device=device):
    model.eval().to(device)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            preds = model(x).argmax(1).cpu().tolist()
            ps.extend(preds)
            ys.extend(y.tolist())
    return {
        "acc":  accuracy_score(ys, ps),
        "bacc": balanced_accuracy_score(ys, ps),
        "y_true": np.array(ys),
        "y_pred": np.array(ps),
    }


print("Training utilities OK")

## Esperimento 1 — Subject-Specific, 4 classi semantiche

Per ogni soggetto in `TEST_SUBJECTS` e ogni modello:
- Split 60/20/20 (train/val/test) sullo stesso soggetto
- Normalizzazione stimata sul train set
- Chance level: **25%** (4 classi bilanciate sui trial)

In [ ]:
# ============================================================
# SUBJECT-SPECIFIC — 4 CLASSI
# ============================================================

N_CLASSES = 4
results_ss_4 = []  # (subject, model, val_acc, val_bacc, test_acc, test_bacc, epochs)

for subj_id in TEST_SUBJECTS:
    ds_train, ds_val, ds_test = make_subject_splits(
        meta, subj_id, labelid2cluster_4, seed=SEED
    )
    print(f"\n── Soggetto {subj_id:02d} │ train={len(ds_train)} val={len(ds_val)} test={len(ds_test)} ──")

    for model_name in MODEL_NAMES:
        t0 = time.time()
        model = build_model(model_name, n_outputs=N_CLASSES)

        model, hist, n_epochs_run = train_model(
            model, ds_train, ds_val, device=device
        )

        val_res  = evaluate(model, ds_val)
        test_res = evaluate(model, ds_test)
        elapsed  = time.time() - t0

        results_ss_4.append({
            "subject":   subj_id,
            "model":     model_name,
            "val_acc":   val_res["acc"],
            "val_bacc":  val_res["bacc"],
            "test_acc":  test_res["acc"],
            "test_bacc": test_res["bacc"],
            "epochs":    n_epochs_run,
            "time_s":    elapsed,
        })

        print(f"  {model_name:<18} │ val_acc={val_res['acc']:.3f}  "
              f"test_acc={test_res['acc']:.3f}  "
              f"test_bacc={test_res['bacc']:.3f}  "
              f"({n_epochs_run} epoche, {elapsed:.0f}s)")

print("\n✓ Subject-specific 4 classi completato")

In [ ]:
# ============================================================
# RISULTATI — tabella riassuntiva 4 classi
# ============================================================

df_ss4 = pd.DataFrame(results_ss_4)

# Media e std per modello
summary_ss4 = df_ss4.groupby("model").agg(
    val_acc_mean=("val_acc",  "mean"),
    val_acc_std= ("val_acc",  "std"),
    test_acc_mean=("test_acc", "mean"),
    test_acc_std= ("test_acc", "std"),
    test_bacc_mean=("test_bacc", "mean"),
    test_bacc_std= ("test_bacc", "std"),
).reset_index()

# Ordina per test_acc
summary_ss4 = summary_ss4.sort_values("test_acc_mean", ascending=False)

print("=== Subject-Specific │ 4 classi semantiche │ Chance = 25% ===")
print(summary_ss4.to_string(index=False, float_format="{:.3f}".format))

# Heatmap test_acc per soggetto × modello
pivot = df_ss4.pivot(index="subject", columns="model", values="test_acc")
# riordina colonne per test_acc medio
pivot = pivot[summary_ss4["model"].tolist()]

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0.0, vmax=1.0, ax=ax,
            linewidths=0.5, cbar_kws={"label": "Test Accuracy"})
ax.set_title("Test Accuracy — Subject-Specific, 4 classi semantiche\n(verde=alto, rosso=basso; chance=0.25)")
ax.set_ylabel("Soggetto")
ax.set_xlabel("")
plt.tight_layout()
plt.savefig(project_root / "figures" / "braindecode_ss4_heatmap.png", dpi=150)
plt.show()
print("Salvato: figures/braindecode_ss4_heatmap.png")

In [ ]:
# Confusion matrix del modello migliore (primo soggetto)
best_model_name = summary_ss4.iloc[0]["model"]
best_subj = TEST_SUBJECTS[0]

print(f"Confusion matrix: {best_model_name} su soggetto {best_subj:02d}")

ds_train, ds_val, ds_test = make_subject_splits(
    meta, best_subj, labelid2cluster_4, seed=SEED
)
model_best = build_model(best_model_name, n_outputs=4)
model_best, _, _ = train_model(model_best, ds_train, ds_val, device=device)
res = evaluate(model_best, ds_test)

cluster_names_4 = ["Azioni", "Cognitivo", "Emozioni", "Oggetti"]
cm = confusion_matrix(res["y_true"], res["y_pred"])
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=cluster_names_4, yticklabels=cluster_names_4, ax=ax)
ax.set_title(f"{best_model_name} — Confusion Matrix normalizzata\n"
             f"Soggetto {best_subj:02d} | Test acc={res['acc']:.3f}")
ax.set_xlabel("Predetto")
ax.set_ylabel("Reale")
plt.tight_layout()
plt.savefig(project_root / "figures" / f"braindecode_{best_model_name}_cm.png", dpi=150)
plt.show()

## Esperimento 2 — Subject-Specific, 5 classi semantiche

Stesso setup ma con 5 cluster — chance level: **20%**.

In [ ]:
# ============================================================
# SUBJECT-SPECIFIC — 5 CLASSI
# ============================================================

N_CLASSES = 5
results_ss_5 = []

for subj_id in TEST_SUBJECTS:
    ds_train, ds_val, ds_test = make_subject_splits(
        meta, subj_id, labelid2cluster_5, seed=SEED
    )
    print(f"\n── Soggetto {subj_id:02d} │ train={len(ds_train)} val={len(ds_val)} test={len(ds_test)} ──")

    for model_name in MODEL_NAMES:
        t0 = time.time()
        model = build_model(model_name, n_outputs=N_CLASSES)
        model, hist, n_epochs_run = train_model(model, ds_train, ds_val, device=device)
        val_res  = evaluate(model, ds_val)
        test_res = evaluate(model, ds_test)
        elapsed  = time.time() - t0

        results_ss_5.append({
            "subject":   subj_id,
            "model":     model_name,
            "val_acc":   val_res["acc"],
            "val_bacc":  val_res["bacc"],
            "test_acc":  test_res["acc"],
            "test_bacc": test_res["bacc"],
            "epochs":    n_epochs_run,
            "time_s":    elapsed,
        })

        print(f"  {model_name:<18} │ val_acc={val_res['acc']:.3f}  "
              f"test_acc={test_res['acc']:.3f}  "
              f"test_bacc={test_res['bacc']:.3f}  "
              f"({n_epochs_run} epoche, {elapsed:.0f}s)")

print("\n✓ Subject-specific 5 classi completato")

In [ ]:
# Tabella riassuntiva 5 classi
df_ss5 = pd.DataFrame(results_ss_5)
summary_ss5 = df_ss5.groupby("model").agg(
    test_acc_mean=("test_acc",  "mean"),
    test_acc_std= ("test_acc",  "std"),
    test_bacc_mean=("test_bacc", "mean"),
    test_bacc_std= ("test_bacc", "std"),
).reset_index().sort_values("test_acc_mean", ascending=False)

print("=== Subject-Specific │ 5 classi semantiche │ Chance = 20% ===")
print(summary_ss5.to_string(index=False, float_format="{:.3f}".format))

## Confronto finale: 4 vs 5 classi

In [ ]:
# ============================================================
# PLOT CONFRONTO 4 vs 5 CLASSI
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

for ax, (df_r, title, chance) in zip(axes, [
    (df_ss4, "4 classi semantiche (chance=25%)", 0.25),
    (df_ss5, "5 classi semantiche (chance=20%)", 0.20),
]):
    order = df_r.groupby("model")["test_acc"].mean().sort_values(ascending=False).index
    means = df_r.groupby("model")["test_acc"].mean()[order]
    stds  = df_r.groupby("model")["test_acc"].std()[order]

    bars = ax.bar(range(len(order)), means, yerr=stds,
                  color=["#2196F3","#4CAF50","#FF9800","#9C27B0","#F44336"][:len(order)],
                  capsize=5, width=0.6, alpha=0.85)
    ax.axhline(chance, ls="--", color="gray", lw=1.5, label=f"Chance ({chance:.0%})")
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, rotation=20, ha="right")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Test Accuracy (media ± std su 5 soggetti)")
    ax.set_title(title)
    ax.legend()
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{mean:.3f}", ha="center", va="bottom", fontsize=9)

plt.suptitle("Braindecode models — Subject-Specific, raw EEG (59ch × 384samples)", y=1.02)
plt.tight_layout()
plt.savefig(project_root / "figures" / "braindecode_comparison_4vs5.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvato: figures/braindecode_comparison_4vs5.png")

## Note e prossimi passi

### Interpretazione risultati
- Risultati **> chance** indicano che il segnale raw contiene struttura semantica decodificabile
- L'approccio end-to-end (segnale grezzo → modello → label) elimina il bias del feature engineering manuale
- Alta varianza tra soggetti è attesa (ε²_soggetto ≈ 0.85 nella nostra analisi)

### Prossimi passi
1. **CBraMod**: aggiornare Python a ≥ 3.11 e aggiungere come 6° modello
2. **Subject-independent**: testare la generalizzazione cross-soggetto (con Instance Normalization da Bomatter 2024)
3. **Data augmentation**: jitter temporale, channel dropout, gaussian noise
4. **Hypergraph**: passare da questo baseline end-to-end al modello DHSLP/DHSLF (Li et al. 2025)
5. **Tuning**: grid search su LR, dropout, dimensioni hidden per il modello migliore